In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("sales_data.csv")
for column in ["order_date", "shipping_date", "delivery_date"]:
    df[column] = pd.to_datetime(df[column])
df.shape

: 

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(before, "->", len(df))

In [ ]:
bad = df["quantity"] > 100
df.loc[bad, "quantity"] = (df.loc[bad, "total_sales"] / df.loc[bad, "unit_price"]).round().astype(int)
diff = (df["total_sales"] - df["quantity"] * df["unit_price"]).abs()
print("고친줄:", bad.sum(), "수량최대:", df["quantity"].max(), "총액안맞는줄:", (diff > 1).sum())

In [ ]:
df["date_error"] = df["delivery_date"] < df["shipping_date"]
df.to_csv("sales_clean.csv", index=False)
print(df["date_error"].sum(), "건,", df.shape)

## ⑤ 배송 기간은 보통 며칠인가?


In [ ]:
valid = df[(~df["date_error"]) & df["delivery_date"].notna() & df["shipping_date"].notna()].copy()
days = (valid["delivery_date"] - valid["shipping_date"]).dt.days

plt.figure(figsize=(8, 4))
plt.hist(days, bins=30, edgecolor="black")
plt.xlabel("Shipping duration (days)")
plt.ylabel("Number of orders")
plt.title("Shipping duration distribution")
plt.show()

print("평균:", days.mean())
print("중앙값:", days.median())

처리 전 평균은 22.05일, 중앙값은 2일로 크게 다른데, 350일 초과건이 오른쪽 끝에 모여 평균을 크게 끌어올렸기 때문이다. 이 값들은 보통 배송으로 보기 어려운 예외값 또는 날짜 입력 오류일 가능성이 있다. 따라서 제외하고 값을 구하면 평균이랑 중앙값이랑 비슷해진다. 배송 기간은 보통 약 2일이다.

## ⑥ 만족도가 낮을수록 반품이 많은가?


In [ ]:
return_rate = (df.assign(is_return=df["status"] == "반품").groupby("satisfaction")["is_return"].mean() * 100)
plt.figure(figsize=(7, 4))
return_rate.plot(kind="bar", color="skyblue", edgecolor="black")
plt.xlabel("Satisfaction")
plt.ylabel("Return rate (%)")
plt.title("Return rate by satisfaction")
plt.xticks(rotation=0)
plt.show()

만족도가 낮을수록 반품이 많다고 보기 어렵다.만족도별 반품 비율은 만족도 1이 7.59%, 2가 11.96%, 3이 7.81%, 4가 8.72%, 5가 6.93%이다. 만족도 2가 가장 높지만, 만족도 1은 만족도 3·4보다 낮아 일정한 감소 또는 증가 추세가 보이지 않는다.

## ⑦ 카테고리 매출 1위의 이유는 건수·단가·수량 중 무엇인가?


In [ ]:
category_summary = (df.groupby("category").agg(건수=("category", "size"), 합=("total_sales", "sum"), 평균_단가=("unit_price", "mean"), 평균_수량=("quantity", "mean")).sort_values("합", ascending=False))
category_summary.round(2)

카테고리 매출 1위는 가구이며 총매출은 약 28.43억 원이다. 가구는 주문 건수 213건과 평균 단가 약 257,322원이 다른 카테고리보다 가장 높다. 평균 수량은 약 51.04개로 가장 높지 않으므로 가구가 매출 1위가 된 이유는 많은 주문 건수와 높은 평균 단가 때문이라고 볼 수 있다.